This notebook shows some design patterns that can be used during ROS2 programming.

Do not run these are examples that you can extend in the excercises

In [ ]:
class MessageReceiver():
    """an object to receive messages, save them do"""
    def basic_save(self,message):
        self.last_message = message
        
saver = MessageReceiver()



In [ ]:
import rclpy
from rclpy.node import Node

from std_msgs.msg import String


class MinimalNode(Node):
    def __init__(self):
        super().__init__("minimal_subscriber")
    

rclpy.init()

minimal_node = MinimalNode()

In [ ]:
subscription = minimal_node.create_subscription(
    String, "wiadomosc", saver.basic_save, 10)
# spin...
## when we receive an message message receiver will take it and save

In [ ]:
saver.last_message

In [ ]:
class ReceiveAndFilter():
    """an object to receive messages, do something with them"""
    summed = 0
    def filter_save(self,message):
        # as an example we will have a sum
        self.summed = self.summed + message.data
        self.last_number = message.data

In [ ]:
from std_msgs.msg import Int32

receiver = ReceiveAndFilter()
subscription = minimal_node.create_subscription(
    Int32, "wiadomosc_z_numerkiem", receiver.filter_save, 10)

In [ ]:
print(receiver.summed)

Feedback loop

In [ ]:
from std_msgs.msg import Int32
from geometry_msgs.msg import Twist


class Reactive_Robot:
    def __init__(self):
        self.publisher = None  # we will have to give the publisher to it
        self.nie_jedz = False
        self.sent_message = None

    def receive_important_message(self, message: Twist):
        "when message comes we publish something"
        if not self.nie_jedz and message.linear.x < 0:
            output_message = Int32()
            output_message.data = 3
            self.publisher.publish(output_message)
            self.sent_message = output_message

    def alarm(self, message):
        stop_message = Int32()
        stop_message.data = 0
        self.publisher.publish(stop_message)
        self.nie_jedz = True


In [ ]:
reactive_robot = Reactive_Robot()
publisher_wiadomosc = minimal_node.create_publisher(std_msgs.Int32,
                                                    "numerki", 10)

reactive_robot.publisher = publisher_wiadomosc

subscription_reactive = minimal_node.create_subscription(
    geometry_msgs.msg.Pose, "poza", reactive_robot.receive_important_message, 10)

Do stuff every second

In [ ]:
from std_msgs.msg import Int32
from geometry_msgs.msg import Pose


class Robot:
    def __init__(self, node):
        self.publisher = node.create_publisher(Int32, "numerki", 10)
        self.subscriber_1 = node.create_subscription(
            Pose, "poza", self.receive_messages_one, 10
        )
        self.subscriber_2 = node.create_subscription(
            Pose, "poza2", self.receive_messages_two, 10
        )
        self.last_message = None
        self.last_message_two = None
        self.timer = node.create_timer(1, self.do_stuff)

    def receive_messages_one(self, message):
        self.last_message = message

    def receive_messages_two(self, message):
        self.last_message_two = message

    def do_stuff(self):
        if self.last_message is None or self.last_message_two is None:
            return
        if self.last_message.position.x < self.last_message_two.position.x:
            output_message = Int32()
            output_message.data = 1
            self.publisher.publish(output_message)


robot = Robot(minimal_node)


In [ ]:
rclpy.spin(minimal_node)